In [48]:

import pandas as pd
# 1. Cargar las dos bases
base = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_con_turnover.xlsx"
)

den = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/denuncias.xlsx"
)

In [50]:
# 2. Suma de delitos por ubigeo
delitos_sum = (
    den.groupby("ubigeo")["cantidad"]
       .sum()
       .reset_index()
       .rename(columns={"cantidad": "suma_delitos"})
)

# 3. Unir por ubigeo
final = base.merge(delitos_sum, on="ubigeo", how="left")

# 4. Guardar base final
final.to_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_final.xlsx",
    index=False
)

In [52]:
import statsmodels.formula.api as smf

# quedarte solo con 2022 (donde existe turnover)
df_2022 = final[final["año"] == 2022]

modelo = smf.ols(
    "turnover_org ~ suma_delitos + C(provincia)",
    data=df_2022
).fit(cov_type="HC1")

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:           turnover_org   R-squared:                       0.168
Model:                            OLS   Adj. R-squared:                  0.141
Method:                 Least Squares   F-statistic:                     1.915
Date:                Thu, 15 Jan 2026   Prob (F-statistic):           2.28e-10
Time:                        19:47:47   Log-Likelihood:                 1145.7
No. Observations:                5010   AIC:                            -1981.
Df Residuals:                    4855   BIC:                            -970.9
Df Model:                         154                                         
Covariance Type:                  HC1                                         
                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 152
  warnings.warn('covariance of constraints does not have full '


In [59]:
import pandas as pd
from linearmodels.iv import IV2SLS

# Cargar base
df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_final.xlsx")

# Quedarse solo con observaciones válidas
df_iv = df[
    df["turnover_org"].notna() &
    df["orden_aparicion"].notna() &
    df["suma_delitos"].notna()
]

# Modelo IV 2SLS
modelo_iv = IV2SLS.from_formula(
    "suma_delitos ~ 1 + [turnover_org ~ orden_aparicion]",
    data=df_iv
).fit(cov_type="robust")

print(modelo_iv.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:           suma_delitos   R-squared:                     -10.263
Estimator:                    IV-2SLS   Adj. R-squared:                -10.265
No. Observations:                5010   F-statistic:                    9.2963
Date:                Thu, Jan 15 2026   P-value (F-stat)                0.0023
Time:                        19:55:32   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                              Parameter Estimates                               
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
Intercept     1.116e+04     2576.2     4.3328     0.0000      6112.7   1.621e+04
turnover_org   -1.7e+05  5.576e+04    -3.049

In [61]:
import pandas as pd
from linearmodels.iv import IV2SLS

# Cargar base
df = pd.read_excel(
    "/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital_final.xlsx"
)

# Restringir explícitamente a 2022
df_iv = df[
    (df["año"] == 2022) &
    (df["turnover_org"].notna()) &
    (df["orden_aparicion"].notna()) &
    (df["suma_delitos"].notna())
]

# Modelo IV con efectos fijos por departamento (region)
modelo_iv_fedep = IV2SLS.from_formula(
    "suma_delitos ~ 1 + C(region) + [turnover_org ~ orden_aparicion]",
    data=df_iv
).fit(cov_type="robust")

print(modelo_iv_fedep.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:           suma_delitos   R-squared:                     -9.0764
Estimator:                    IV-2SLS   Adj. R-squared:                -9.1269
No. Observations:                5010   F-statistic:                    438.86
Date:                Thu, Jan 15 2026   P-value (F-stat)                0.0000
Time:                        19:57:34   Distribution:                 chi2(25)
Cov. Estimator:                robust                                         
                                                                              
                                     Parameter Estimates                                      
                            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------------------
Intercept                   2.257e+04  1.318e+04     1.7123     0.0868     -3263.6 